In [20]:
from bs4 import XMLParsedAsHTMLWarning
import pandas as pd
import warnings
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

import sys
sys.path.append("..")  # src 모듈을 import 하기 위해 상위 폴더를 경로에 추가

from src.edgar_client import fetch_filing_html
from src.parser import find_schedule_of_investments_tables, _row_cells, parse_filing_html

url = "https://www.sec.gov/Archives/edgar/data/1736035/000173603526000010/bxsl-20260331.htm"
html = fetch_filing_html(url)

tables = find_schedule_of_investments_tables(html)
print(f"찾은 테이블 개수: {len(tables)}")

찾은 테이블 개수: 86


In [21]:
# from src.parser import find_schedule_of_investments_tables, _table_as_of_date
# from src.edgar_client import extract_period_end_date

# tables = find_schedule_of_investments_tables(html)
# for i, table in enumerate(tables):
#     print(i, _table_as_of_date(table))

In [22]:
# from src.pipeline import build_time_series, save_to_sqlite
# from config import BXSL_CIK

# parsed, unmatched = build_time_series(BXSL_CIK, form_types=["10-Q", "10-K"], limit=8)
# print(parsed.shape, unmatched.shape)

# save_to_sqlite(parsed, unmatched)

In [23]:
# from src.edgar_client import get_recent_filings, build_document_url, fetch_filing_html
# from src.parser import find_schedule_of_investments_tables, _table_as_of_date, _row_cells
# from config import BXSL_CIK

# # 2024-06-30 filing만 다시 열어서 Aevex 관련 원본 행 확인
# filings = get_recent_filings(BXSL_CIK, form_types=["10-Q"], limit=8)
# target = [f for f in filings if f["filing_date"] == "2024-08-07"][0]
# url = build_document_url(target["cik"], target["accession_number"], target["primary_document"])
# html_2024q2 = fetch_filing_html(url)

# tables_2024q2 = find_schedule_of_investments_tables(html_2024q2)
# current_2024q2 = [t for t in tables_2024q2 if _table_as_of_date(t) == "2024-06-30"]

# for table in current_2024q2:
#     for tr in table.find_all("tr"):
#         cells = _row_cells(tr)
#         if any("Aevex" in c for c in cells):
#             print(cells)

In [24]:
# parsed[parsed["investment_name"] == "Aevex Holdings, LLC"][
#     ["investment_name", "fair_value", "period_end_date", "acquisition_date"]
# ].sort_values("period_end_date")

In [25]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")
import sqlite3
import pandas as pd

from src.metrics import clean_numeric_columns, detect_events
from src.normalizer import normalize_entity_name

conn = sqlite3.connect("../data/bdc_tracker.db")
positions = pd.read_sql("SELECT * FROM positions", conn)
conn.close()

positions_clean = clean_numeric_columns(positions)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [26]:
from src.normalizer import normalize_entity_name

print(normalize_entity_name("BP Alpha Holdings, L.P."))
print(normalize_entity_name("BP Alpha Holdings, LP"))

bp alpha holdings
bp alpha holdings


In [27]:
from src.metrics import detect_events

events = detect_events(positions_clean)
print(events["event_type"].value_counts())

box_events = events[events["investment_name"].str.contains("Box Co-Invest", case=False, na=False)]
print(box_events[["investment_name", "event_type", "period_end_date", "fair_value"]])

event_type
new_entry    643
markup       219
exit         141
markdown     102
Name: count, dtype: int64
                                       investment_name event_type  \
49   Box Co-Invest Blocker, LLC - (BP Alpha Holding...  new_entry   
255  Box Co-Invest Blocker, LLC - (BP Alpha Holding...  new_entry   
478  Box Co-Invest Blocker, LLC - (BP Alpha Holding...   markdown   
492  Box Co-Invest Blocker, LLC - (BP Alpha Holding...   markdown   
595  Box Co-Invest Blocker, LLC - (BP Alpha Holding...   markdown   

    period_end_date  fair_value  
49       2024-09-30       155.0  
255      2024-09-30       102.0  
478      2024-12-31        16.0  
492      2024-12-31         0.0  
595      2025-03-31         0.0  


In [28]:
from src.metrics import detect_events

events = detect_events(positions_clean)
print(events["event_type"].value_counts())

box_events = events[events["investment_name"].str.contains("Box Co-Invest", case=False, na=False)]
print(box_events[["investment_name", "event_type", "period_end_date", "fair_value"]])

event_type
new_entry    643
markup       219
exit         141
markdown     102
Name: count, dtype: int64
                                       investment_name event_type  \
49   Box Co-Invest Blocker, LLC - (BP Alpha Holding...  new_entry   
255  Box Co-Invest Blocker, LLC - (BP Alpha Holding...  new_entry   
478  Box Co-Invest Blocker, LLC - (BP Alpha Holding...   markdown   
492  Box Co-Invest Blocker, LLC - (BP Alpha Holding...   markdown   
595  Box Co-Invest Blocker, LLC - (BP Alpha Holding...   markdown   

    period_end_date  fair_value  
49       2024-09-30       155.0  
255      2024-09-30       102.0  
478      2024-12-31        16.0  
492      2024-12-31         0.0  
595      2025-03-31         0.0  


In [29]:
from src.metrics import detect_events

events = detect_events(positions_clean)
print(events["event_type"].value_counts())

event_type
new_entry    643
markup       219
exit         141
markdown     102
Name: count, dtype: int64


In [30]:
from src.ai_classifier import classify_unmatched_rows

sample = unmatched.head(15)
classified_sample = classify_unmatched_rows(sample)
classified_sample

Classifying rows 0-14 of 15...


,investment_type,investment_name,counterparty,currency,par_amount,cost,fair_value,pct_of_net_assets,maturity_date,acquisition_date,notes,period_end_date,filing_date,accession_number
0,equity_or_debt_with_gaps,Zoro - Series A Preferred Shares,NaN,NaN,88,85,136,0.00,NaN,11/22/2022,Preferred equity with PIK interest (SOFR + 9.5...,2026-03-31,2026-05-07,0001736035-26-000010
1,equity_or_debt_with_gaps,Blackstone Donegal Holdings LP - LP Interests ...,NaN,NaN,1,NaN,5221,0.09,NaN,1/5/2021,LP interest position missing cost field,2026-03-31,2026-05-07,0001736035-26-000010
2,other,Other Cash and Cash Equivalents,NaN,NaN,NaN,305794,305794,5.01,NaN,NaN,Cash equivalents aggregate position,2026-03-31,2026-05-07,0001736035-26-000010
3,fx_forward,NaN,"Wells Fargo Bank, N.A.",USD/CAD,NaN,NaN,974,NaN,6/25/2026,NaN,"FX forward: USD 65,701 / CAD 90,000",2026-03-31,2026-05-07,0001736035-26-000010
4,fx_forward,NaN,"Wells Fargo Bank, N.A.",USD/EUR,NaN,NaN,200,NaN,6/25/2026,NaN,"FX forward: USD 75,631 / EUR 65,250",2026-03-31,2026-05-07,0001736035-26-000010
5,fx_forward,NaN,"Wells Fargo Bank, N.A.",USD/NOK,NaN,NaN,46,NaN,6/25/2026,NaN,"FX forward: USD 2,783 / NOK 26,704",2026-03-31,2026-05-07,0001736035-26-000010
6,fx_forward,NaN,"Wells Fargo Bank, N.A.",USD/DKK,NaN,NaN,5,NaN,6/25/2026,NaN,"FX forward: USD 1,856 / DKK 11,958",2026-03-31,2026-05-07,0001736035-26-000010
7,fx_forward,NaN,"Wells Fargo Bank, N.A.",USD/GBP,NaN,NaN,611,NaN,6/25/2026,NaN,"FX forward: USD 62,560 / GBP 47,000",2026-03-31,2026-05-07,0001736035-26-000010
8,fx_forward,NaN,"Wells Fargo Bank, N.A.",USD/SEK,NaN,NaN,416,NaN,6/25/2026,NaN,"FX forward: USD 23,347 / SEK 217,760",2026-03-31,2026-05-07,0001736035-26-000010
9,equity_or_debt_with_gaps,Zoro - Series A Preferred Shares,NaN,NaN,122,118,182,0.00,NaN,11/22/2022,Preferred equity with PIK interest (SOFR + 9.5...,2025-12-31,2026-02-25,0001736035-26-000004


In [31]:
classified_sample = classify_unmatched_rows(unmatched)
classified_sample[classified_sample["investment_type"] == "fx_forward"][
    ["counterparty", "currency_purchased_amount", "currency_purchased_code",
     "currency_sold_amount", "currency_sold_code", "settlement_date", "unrealized_gain_loss"]
]

[SAMPLE MODE] Processing 15 of 456 rows. Pass sample_size=None to run the full dataset.
Classifying rows 0-14 of 15...
  Batch cost: $0.0569 (running total: $0.0569)

Total API cost for this run: $0.0569


,counterparty,currency_purchased_amount,currency_purchased_code,currency_sold_amount,currency_sold_code,settlement_date,unrealized_gain_loss
3,"Wells Fargo Bank, N.A.",65701.0,USD,90000.0,CAD,6/25/2026,974.0
4,"Wells Fargo Bank, N.A.",75631.0,USD,65250.0,EUR,6/25/2026,200.0
5,"Wells Fargo Bank, N.A.",2783.0,USD,26704.0,NOK,6/25/2026,46.0
6,"Wells Fargo Bank, N.A.",1856.0,USD,11958.0,DKK,6/25/2026,5.0
7,"Wells Fargo Bank, N.A.",62560.0,USD,47000.0,GBP,6/25/2026,611.0
8,"Wells Fargo Bank, N.A.",23347.0,USD,217760.0,SEK,6/25/2026,416.0
12,"Wells Fargo Bank, N.A.",58283.0,USD,80000.0,CAD,03/25/2026,-255.0
13,"Wells Fargo Bank, N.A.",76983.0,USD,65250.0,EUR,03/25/2026,134.0
14,"Wells Fargo Bank, N.A.",2626.0,USD,26704.0,NOK,03/25/2026,-22.0


In [32]:
print(classified_sample["investment_type"].value_counts())

investment_type
fx_forward                  9
equity_or_debt_with_gaps    4
money_market_fund           2
Name: count, dtype: int64


In [33]:
swap_rows = unmatched[
    unmatched["raw_tokens"].astype(str).str.contains("SMBC|Interest Rate Swap|Notes", case=False, na=False)
]
print(swap_rows.shape)

if len(swap_rows) > 0:
    classified_swaps = classify_unmatched_rows(swap_rows, sample_size=None)
    classified_swaps[["investment_type", "counterparty", "notional_amount", "unrealized_gain_loss", "maturity_date"]]

(0, 7)


In [34]:
# 현재 로드된 html(2026-05-07 filing)에서 "Interest Rate Swap" 텍스트 자체가 있는지 확인
print("Interest Rate Swap" in html)
print("ADDITIONAL INFORMATION" in html)

True
True
